# 🚀 MaroMart AI - One Click Training from GitHub (Optimized)
Notebook này tự động clone code từ GitHub, cấu hình thư viện đúng phiên bản, tự động sinh 3000+ dữ liệu tự nhiên nhu cầu người dùng, huấn luyện model ViT5 siêu tốc (tối ưu VRAM) và lưu kết quả thẳng vào Google Drive.

**Lưu ý:** Để chạy được phiên bản tối ưu mới nhất (F1 > 85%, Dynamic Padding), hãy đảm bảo bạn đã đẩy các thay đổi cục bộ lên GitHub của bạn, HOẶC chạy Notebook này - nó sẽ tự động áp dụng các tối ưu hóa mới nhất trực tiếp!

In [ ]:
# 1. Kết nối Google Drive để lưu trữ lâu dài
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_PATH = "/content/drive/MyDrive/Temo/search"
for p in [DRIVE_PATH, f"{DRIVE_PATH}/file_train", f"{DRIVE_PATH}/dataset"]:
    os.makedirs(p, exist_ok=True)
print(f"✅ Google Drive sẵn sàng: {DRIVE_PATH}")

In [ ]:
# 2. Clone Repo từ GitHub của bạn
%cd /content
!rm -rf Search_temo_AI
!git clone https://github.com/hoangnqjl/Search_temo_AI.git
%cd Search_temo_AI
print("✅ Clone thành công!")

In [ ]:
# 3. Cài đặt thư viện đúng phiên bản ghim ổn định
!pip install -q -r requirements.txt

import transformers, tokenizers, sentencepiece
print(f"✅ Thư viện sẵn sàng: transformers={transformers.__version__}")

### 4. Áp dụng Tối ưu hóa & Sinh Dữ liệu Nhu cầu Tự nhiên
Đảm bảo sinh dữ liệu trước khi train để tăng kích thước dataset từ 207 lên 4,155 dòng.

In [ ]:
# 4.1 Tự động sinh dữ liệu 4,155 dòng mô tả nhu cầu tự nhiên của con người
print(">>> Khởi động bộ tạo dữ liệu nhu cầu thực tế...")
!python build/generate_synthetic_data.py

# Đồng bộ tệp dữ liệu đã sinh lên Drive để lưu trữ dự phòng
import shutil
if os.path.exists("data/augmented_dataset.csv"):
    shutil.copy("data/augmented_dataset.csv", f"{DRIVE_PATH}/dataset/augmented_dataset.csv")
    print(f"✅ Đã đồng bộ augmented_dataset.csv lên Drive: {DRIVE_PATH}/dataset")

### 5. Khởi chạy Huấn luyện Tinh chỉnh ViT5
Mô hình sẽ huấn luyện với Dynamic Padding (siêu tiết kiệm VRAM) và lưu model tốt nhất có F1 > 85% vào Google Drive.

In [ ]:
# 5.1 Bắt đầu quá trình huấn luyện và xuất báo cáo Recall, Precision, Accuracy, F1
# Kết quả .json và mô hình tốt nhất (best_model) sẽ lưu trực tiếp vào Drive của bạn tại Temo/search/file_train
!python build/train_vit5_grid.py

---
## 🧪 Phần 6: Thử nghiệm nhanh mô hình (Inference Demo)
Sau khi mô hình chạy xong, bạn có thể chạy thử trực tiếp để kiểm tra chất lượng dịch thuật nhu cầu.

In [ ]:
# 6.1 Demo: Phân tích ý định người dùng (Sinh từ khóa tên sản phẩm tương ứng)
import sys
if "." not in sys.path: sys.path.append(".")

from build.demo import run_demo

# Nhập câu mô tả nhu cầu bất kỳ ở đây để AI nhận diện ra tên sản phẩm
run_demo("Tớ đang cần tìm cái điện thoại cũ pin trâu ngoại hình đẹp tầm 5 triệu để tặng em gái đi học")